In [ ]:
import json
import logging
import os
from pathlib import Path

from datasets import DatasetDict, Dataset
from transformers import AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# =========================
# CONFIG
# =========================
MODEL_NAME = "microsoft/deberta-base"
CACHE_PATH = "./tokenized_cuad"

MAX_LENGTH = 384
DOC_STRIDE = 64


In [ ]:
# =========================
# LOAD DATASET
# =========================
def load_cuad_dataset():
    candidates = [
        os.environ.get("CUAD_JSON", ""),
        "./CUAD_v1.json",
    ]

    json_path = next((p for p in candidates if p and Path(p).exists()), None)

    if json_path is None:
        raise FileNotFoundError("CUAD_v1.json not found")

    logger.info(f"Loading CUAD dataset from: {json_path}")

    with open(json_path, encoding="utf-8") as f:
        raw = json.load(f)

    rows = {"id": [], "question": [], "context": [], "answers": []}

    for doc in raw["data"]:
        for para in doc["paragraphs"]:
            context = para["context"]

            for qa in para["qas"]:
                rows["id"].append(qa["id"])
                rows["question"].append(qa["question"].strip())
                rows["context"].append(context)

                if qa.get("is_impossible") or not qa["answers"]:
                    rows["answers"].append({"text": [], "answer_start": []})
                else:
                    rows["answers"].append({
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    })

    flat = Dataset.from_dict(rows)
    logger.info(f"Loaded {len(flat)} QA pairs")

    split = flat.train_test_split(test_size=0.10, seed=42)
    train_val = split["train"]
    test = split["test"]

    split = train_val.train_test_split(test_size=0.111, seed=42)
    train = split["train"]
    val = split["test"]

    dataset = DatasetDict({
        "train": train,
        "validation": val,
        "test": test,
    })

    logger.info(f"Final sizes → train: {len(train)}, val: {len(val)}, test: {len(test)}")

    return dataset


In [ ]:
# =========================
# SAFE PREPROCESS FUNCTION
# =========================
def preprocess_for_qa(examples, tokenizer):
    tokenized = tokenizer(
        [q.strip() for q in examples["question"]],
        examples["context"],
        max_length=384,
        truncation="only_second",
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")
    answers = examples["answers"]

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_mapping):

        if offsets is None:
            start_positions.append(0)
            end_positions.append(0)
            continue

        sequence_ids = tokenized.sequence_ids(i)
        if sequence_ids is None:
            start_positions.append(0)
            end_positions.append(0)
            continue

        sample_idx = sample_map[i]
        answer = answers[sample_idx]

        try:
            ctx_start = next(j for j, s in enumerate(sequence_ids) if s == 1)
            ctx_end = len(sequence_ids) - 1 - next(
                j for j, s in enumerate(reversed(sequence_ids)) if s == 1
            )
        except StopIteration:
            start_positions.append(0)
            end_positions.append(0)
            continue

        if not answer["answer_start"]:
            start_positions.append(0)
            end_positions.append(0)
            continue

        char_start = answer["answer_start"][0]
        char_end = char_start + len(answer["text"][0])

        if offsets[ctx_start] is None or offsets[ctx_end] is None:
            start_positions.append(0)
            end_positions.append(0)
            continue

        if offsets[ctx_start][0] > char_end or offsets[ctx_end][1] < char_start:
            start_positions.append(0)
            end_positions.append(0)
            continue

        token_start = ctx_start
        while (
            token_start <= ctx_end
            and offsets[token_start] is not None
            and offsets[token_start][0] is not None
            and offsets[token_start][0] <= char_start
        ):
            token_start += 1

        token_end = ctx_end
        while (
            token_end >= ctx_start
            and offsets[token_end] is not None
            and offsets[token_end][1] is not None
            and offsets[token_end][1] >= char_end
        ):
            token_end -= 1

        start_positions.append(max(ctx_start, token_start - 1))
        end_positions.append(min(ctx_end, token_end + 1))

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized


In [ ]:
# =========================
# MAIN EXECUTION
# =========================
def main():

    if os.path.exists(CACHE_PATH):
        logger.info("✅ Tokenized dataset already exists. Skipping preprocessing.")
        return

    dataset = load_cuad_dataset()

    logger.info("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    logger.info("Starting preprocessing (this will take a few minutes)...")

    tokenized = dataset.map(
        preprocess_for_qa,
        fn_kwargs={"tokenizer": tokenizer},
        batched=True,
        num_proc=None,
        remove_columns=dataset["train"].column_names,
    )
    logger.info("Saving tokenized dataset...")
    tokenized.save_to_disk(CACHE_PATH)

    logger.info("✅ DONE! Dataset saved at: " + CACHE_PATH)


if __name__ == "__main__":
    main()

In [18]:
from datasets import load_from_disk
from transformers import AutoTokenizer

# load
dataset = load_from_disk("./tokenized_cuad_old")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

print(dataset)

# pick random sample
sample = dataset["train"][0]

print("\nKeys:", sample.keys())

print("\nStart:", sample["start_positions"])
print("End:", sample["end_positions"])

# decode full input
decoded = tokenizer.decode(sample["input_ids"])

print("\n--- FULL TEXT ---\n")
print(decoded[:1000])

# extract predicted answer span
start = sample["start_positions"]
end = sample["end_positions"]

if start != 0 or end != 0:
    answer_tokens = sample["input_ids"][start:end+1]
    answer = tokenizer.decode(answer_tokens)
    print("\n--- EXTRACTED ANSWER ---\n")
    print(answer)
else:
    print("\nNo answer (correct for negative sample)")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 1089767
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 136452
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 133662
    })
})

Keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'])

Start: 0
End: 0

--- FULL TEXT ---

[CLS]Highlight the parts (if any) of this contract related to "Non-Disparagement" that should be reviewed by a lawyer. Details: Is there a requirement on a party not to disparage the counterparty?[SEP]1.

2.

2.1

2.2

3.

3.1

3.2

4.

4.1



EXHIBIT 1.1 Strategic Alliance Agreement

This Agreement is made and entered into this 30th day of June 2017,

Between:

In [11]:
from datasets import load_from_disk
from transformers import AutoTokenizer
import random

dataset = load_from_disk("./tokenized_cuad")
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

found = False

for _ in range(1000):
    sample = dataset["train"][random.randint(0, 100000)]

    start = sample["start_positions"]
    end = sample["end_positions"]

    if start != 0 or end != 0:
        print("\nFOUND POSITIVE SAMPLE\n")

        answer = tokenizer.decode(sample["input_ids"][start:end+1])
        full_text = tokenizer.decode(sample["input_ids"])

        print("Start:", start)
        print("End:", end)
        print("\nAnswer:\n", answer)
        print("\nContext snippet:\n", full_text[:500])

        found = True
        break

if not found:
    print("⚠️ No positive samples found — problem!")


FOUND POSITIVE SAMPLE

Start: 333
End: 369

Answer:
 IMedicor will provide a warrant to purchase 2 million shares of common stock to USA MCO to offset any up-front marketing expense incurred by USA MCO in this project.

Context snippet:
 [CLS]Highlight the parts (if any) of this contract related to "Revenue/Profit Sharing" that should be reviewed by a lawyer. Details: Is one party required to share revenue or profit with the counterparty for any technology, goods, or services?[SEP] SEC required filings that from time to time will include mention of the Strategic Alliance between iMedicor and USA MCO ●IMedicor shall provide access to the iMedicor system, training and customer support as required. ●USA MCO will have the option to 


In [1]:
import json

with open("./CUAD_v1.json") as f:
    raw = json.load(f)

total, answerable = 0, 0
for doc in raw["data"]:
    for para in doc["paragraphs"]:
        for qa in para["qas"]:
            total += 1
            if not qa.get("is_impossible") and qa["answers"]:
                answerable += 1

print(f"Total QA pairs : {total}")
print(f"Answerable     : {answerable}  ({answerable/total*100:.1f}%)")
print(f"Unanswerable   : {total - answerable}")

Total QA pairs : 20910
Answerable     : 6702  (32.1%)
Unanswerable   : 14208
